## Topic: RunnableParallel 

### Agenda
- 1. Introduction of RunnableParallel

- 2. Practical Example 

- 3. Summary


### 1. Introduction of RunnableParallel

- Definition:
    - RunnableParallel is a runnable primitive that allows multiple runnables to execute in parallel.

    - Each runnable receives the same input and processes it independently, producing a dictionary of outputs.


- Key Concept:
    - RunnableParallel is a LangChain Runnable that runs multiple sub-chains simultaneously (in parallel) on the same input, and merges their outputs into a single dictionary

    
    - All sub-chains are executed concurrently.
    
    - They do not depend on each other — each receives the same input.
    
    - The final result is a dictionary with keys corresponding to the sub-chain names.


In [ ]:
"""                             - Visual model
                            =================================
- RunnableParallel is used when we want to execute multiple independent Runnables using the same input.

                 ┌──→ Runnable 1 → Output 1
                 │
Input ───────────┼──→ Runnable 2 → Output 2
                 │
                 └──→ Runnable 3 → Output 3

======================================================================================
- Example: 

                    ┌──► [Chain A: Summarization] ──► "summary" (str)
                    │
[Single Input] ─────┼──► [Chain B: Keywords]    ──► "keywords" (list)
                    │
                    └──► [Chain C: Sentiment]   ──► "sentiment" (str)
                            
                 Output: {"summary": "...", "keywords": [...], "sentiment": "Positive"}



"""

In [ ]:
"""             - RunnableParallel INTERNAL FLOW 

┌─────────────────────────────────────────────────────────────┐
│           RunnableParallel INTERNAL FLOW                    │
│                                                             │
│  1. RECEIVE INPUT                                           │
│     input = {"text": "LangChain is great!"}                 │
│                                                             │
│  2. FAN-OUT: Send the same input to ALL sub-chains          │
│     ┌──► summary_chain.invoke(input)   ──► Thread 1         │
│     ├──► sentiment_chain.invoke(input) ──► Thread 2         │
│     └──► keywords_chain.invoke(input)  ──► Thread 3         │
│                                                             │
│  3. WAIT for ALL to finish (the slowest one determines      │
│     total time)                                             │
│                                                             │
│  4. FAN-IN: Collect each output into a dictionary           │
│     output = {                                              │
│         "summary": "...",                                   │
│         "sentiment": "Positive",                            │
│         "keywords": ["LangChain", "LLM"]                    │
│     }                                                       │
│                                                             │
│  5. RETURN the dictionary                                   │
└─────────────────────────────────────────────────────────────┘

"""

#### 2. Practical Example  
- Idea:
    - step1: topic: AI 

    - step2.1: LLM1 
        - input: topic
        - process: topic | model | parser
        - output: twitter post generate

    - step2.2: LLM2
        - input: topic
        - process: topic | model | parser
        - output: linkedin post generate


- Key note:
    - RunnableParallel is define into python dictionary format.

In [ ]:
### Example 1: 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel

from dotenv import load_dotenv
load_dotenv()

# prompt1
prompt1 = PromptTemplate(
    template = "Generate a tweet about {topic}",
    input_variables = ["topic"]
)
# prompt2
prompt2 = PromptTemplate(
    template = "Generate a linkedin about {topic}",
    input_variables = ["topic"]
)


# Define model
model = ChatOpenAI()

# parser
parser = StrOutputParser()


# Define RunnableParallel to create a parallel chain
parallel_chain = RunnableParallel({
    'tweet': RunnableSequence(prompt1 | model | parser),
    'linkedin': RunnableSequence(prompt2 | model | parser)
})

# invoke the parallel chain
response = parallel_chain.invoke(
    {
        "topic": "RAG"
    }
)

print(f"Entire response:\n {response}")

print(f"tweet post: \n{response['tweet']}")
print(f"Linkedin post: \n{response['linkedin']}")

In [ ]:
# Example 2 — Multi-Aspect Analysis 


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel

llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
parser = StrOutputParser()

# Define 5 independent analysis chains
summary_chain = ChatPromptTemplate.from_template("Summarize in 3 sentences:\n{text}") | llm | parser

entities_chain = (
    ChatPromptTemplate.from_template(
        "Extract named entities (people, orgs, locations) as JSON from:\n{text}"
    )
    | llm
    | JsonOutputParser()
)

tone_chain = ChatPromptTemplate.from_template("Analyze tone and sentiment:\n{text}") | llm | parser

risk_chain = ChatPromptTemplate.from_template("Identify risks or red flags:\n{text}") | llm | parser

action_chain = ChatPromptTemplate.from_template("List action items as bullets:\n{text}") | llm | parser

# Run everything in parallel
analyzer = RunnableParallel(
    executive_summary=summary_chain,
    entities=entities_chain,
    sentiment_tone=tone_chain,
    risks=risk_chain,
    action_items=action_chain
)

document = """...your lengthy document..."""
result = analyzer.invoke({"text": document})

# Access each result by key
print(result["executive_summary"])
print(result["entities"])
print(result["sentiment_tone"])
# all computed simultaneously!

### 3. The Complete Summary

In [ ]:
"""    - Summary of RunnableParallel

┌────────────────────────────────────────────────────────────────────┐
│                        RunnableParallel                              │
│                                                                      │
│  WHAT: Runs multiple independent Runnables SIMULTANEOUSLY.          │
│  WHY:  Eliminate sequential latency for independent tasks.          │
│                                                                      │
│  SYNTAX (Implicit / Most Common):                                    │
│     parallel = {                                                     │
│         "summary": summary_chain,                                    │
│         "keywords": keyword_chain,                                   │
│         "sentiment": sentiment_chain                                 │
│     }                                                               │
│     result = parallel.invoke(single_input)                          │
│     # Returns: {"summary": ..., "keywords": ..., "sentiment": ...}  │
│                                                                      │
│  KEY CONSTRAINT:                                                     │
│     - All branches get THE SAME input                                │
│     - If input needs splitting, use lambdas/itemgetter per branch    │
│     - Total time = time of the SLOWEST branch, not the sum           │
│                                                                      │
│  INTERNALS:                                                            │
│     - Uses asyncio.gather() / ThreadPoolExecutor under the hood      │
│     - Non-blocking network calls run concurrently                    │
│                                                                      │
│  USE CASES:                                                           │
│     ├── Multi-source RAG retrieval (tech docs + legal docs)          │
│     ├── Multi-model consensus (GPT + Claude + Gemini)                │
│     ├── Multi-aspect analysis (sentiment + entities + summary)       │
│     ├── Data enrichment (.assign() with parallel Runnables)          │
│     └── Nested parallel (analyzing multiple business units)          │
│                                                                      │
│  GOLDEN RULE:                                                         │
│  "Whenever your steps are INDEPENDENT, combine them with a dict.      │
│   If you have to call 3 APIs that don't need each other,             │
│   you're wasting 3x of your user's patience if you use `|`."         │
└──────────────────────────────────────────────────────────────────────┘
 



"""